# Phase 2 (Agent 4): cross-encoder re-ranking at full scale (E2/E3)

Scores (query, document) pairs with `cross-encoder/ms-marco-MiniLM-L-6-v2` for every evaluated query on the 8 BEIR datasets. Quora is limited to about 2,100 of its 10k queries to keep the file small.

The candidate pools are:
- BM25 top-100;
- the fixed hybrid top-100 for each dense model;
- the fused top-10 of the alpha grid.

The CPU run in the project scores only 150 queries per dataset.

**Run `phase2_dense_runs.ipynb` first**, so the dense runs are in `MyDrive/IR_Lexical_Gap/colab_runs`. If they are missing, only the BM25 top-100 is scored.

1. *Runtime > Change runtime type > T4 GPU*.
2. *Runtime > Run all*. It takes about 45-75 min. It is resumable: if Colab disconnects, run all again.
3. The last cell downloads `phase2_ce_runs.zip` (about 40-80 MB). Upload it to the Claude chat.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
!pip -q install sentence-transformers PyStemmer
import os
for sub in ('data', 'src'):
    os.makedirs('/content/proj/' + sub, exist_ok=True)

In [ ]:
%%bash
cd /content/proj/data
for d in nfcorpus scifact fiqa scidocs arguana trec-covid quora webis-touche2020; do
  [ -f $d/corpus.jsonl ] || (wget -q https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/$d.zip && unzip -q -o $d.zip && rm $d.zip)
done; ls

In [ ]:
%%writefile /content/proj/src/common.py
"""Shared utilities: data loading, tokenisation/indexing, evaluation, run I/O.

All randomness is driven by SEED. Paths are relative to the project root.
"""
import json
import os
import re
import pickle
from collections import defaultdict

import numpy as np
import scipy.sparse as sp
import Stemmer
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA = os.path.join(ROOT, "data")
CACHE = os.path.join(ROOT, "cache")
RES = os.path.join(ROOT, "results")
FIG = os.path.join(ROOT, "figures")
for _d in (CACHE, RES, FIG):
    os.makedirs(_d, exist_ok=True)

DATASETS = ["nfcorpus", "scifact", "fiqa", "scidocs", "arguana",
            "trec-covid", "quora", "webis-touche2020"]
if os.environ.get("IRGAP_DATASETS"):  # optional subset for smoke tests only; verdict rules assume all 8
    DATASETS = os.environ["IRGAP_DATASETS"].split(",")
CORE = ["nfcorpus", "scifact", "fiqa"]  # the three datasets of the original proposal
DOMAIN = {"nfcorpus": "biomedical", "scifact": "scientific", "fiqa": "financial",
          "scidocs": "scientific", "arguana": "argument", "trec-covid": "biomedical",
          "quora": "general (dup. questions)", "webis-touche2020": "argument"}
PRETTY = {"nfcorpus": "NFCorpus", "scifact": "SciFact", "fiqa": "FiQA-2018",
          "scidocs": "SciDocs", "arguana": "ArguAna", "trec-covid": "TREC-COVID",
          "quora": "Quora", "webis-touche2020": "Touche-2020"}

# Lucene's default English stopword set (what Pyserini/Anserini use for BEIR BM25).
LUCENE_STOP = set("""a an and are as at be but by for if in into is it no not of on or
such that the their then there these they this to was will with""".split())

TOKEN_RE = re.compile(r"\w+", re.UNICODE)
_stemmer = Stemmer.Stemmer("porter")


# ----------------------------------------------------------------------------- data
def load_dataset(name):
    d = os.path.join(DATA, name)
    doc_ids, texts = [], []
    with open(os.path.join(d, "corpus.jsonl")) as f:
        for line in f:
            o = json.loads(line)
            doc_ids.append(str(o["_id"]))
            texts.append((o.get("title") or "") + " " + (o.get("text") or ""))
    queries = {}
    with open(os.path.join(d, "queries.jsonl")) as f:
        for line in f:
            o = json.loads(line)
            queries[str(o["_id"])] = o["text"]
    qrels = defaultdict(dict)
    with open(os.path.join(d, "qrels", "test.tsv")) as f:
        next(f)
        for line in f:
            q, doc, s = line.rstrip("\n").split("\t")
            qrels[q][doc] = int(s)
    # keep only queries that have at least one positive judgement (BEIR/trec_eval practice)
    qrels = {q: r for q, r in qrels.items() if any(v > 0 for v in r.values()) and q in queries}
    queries = {q: queries[q] for q in qrels}
    return doc_ids, texts, queries, qrels


def tokenize(text):
    return TOKEN_RE.findall(text.lower())


# ----------------------------------------------------------------------------- index
class Index:
    """Raw count matrix built once; stem/stop variants derived by column mapping."""

    def __init__(self, name):
        self.name = name
        path = os.path.join(CACHE, f"{name}_index.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f:
                self.__dict__.update(pickle.load(f))
            return
        doc_ids, texts, queries, qrels = load_dataset(name)
        cv = CountVectorizer(lowercase=True, token_pattern=r"(?u)\w+", dtype=np.int32)
        C = cv.fit_transform(texts).tocsr()
        raw_vocab = cv.get_feature_names_out().tolist()
        self.doc_ids, self.queries, self.qrels = doc_ids, queries, qrels
        self.raw_vocab = raw_vocab
        self.C_raw = C
        with open(path, "wb") as f:
            pickle.dump(self.__dict__, f, protocol=4)

    def variant(self, stem=True, stop=True):
        """Return (doc-term count matrix, vocab dict term->col, analyser fn)."""
        key = (stem, stop)
        if not hasattr(self, "_variants"):
            self._variants = {}
        if key in self._variants:
            return self._variants[key]
        raw = self.raw_vocab
        terms = _stemmer.stemWords(raw) if stem else raw
        keep = np.array([t not in LUCENE_STOP for t in raw]) if stop else np.ones(len(raw), bool)
        vocab = {}
        rows, cols = [], []
        for i, (t, k) in enumerate(zip(terms, keep)):
            if not k:
                continue
            j = vocab.setdefault(t, len(vocab))
            rows.append(i)
            cols.append(j)
        M = sp.csr_matrix((np.ones(len(rows), np.int32), (rows, cols)), shape=(len(raw), len(vocab)))
        C = (self.C_raw @ M).tocsr()
        C.sort_indices()

        def analyse(text, _stem=stem, _stop=stop):
            toks = tokenize(text)
            if _stop:
                toks = [t for t in toks if t not in LUCENE_STOP]
            if _stem:
                toks = _stemmer.stemWords(toks)
            return toks

        self._variants[key] = (C, vocab, analyse)
        return self._variants[key]


def query_matrix(qtexts, vocab, analyse, binary=False):
    rows, cols, vals = [], [], []
    for i, t in enumerate(qtexts):
        counts = defaultdict(int)
        for tok in analyse(t):
            j = vocab.get(tok)
            if j is not None:
                counts[j] += 1
        for j, c in counts.items():
            rows.append(i)
            cols.append(j)
            vals.append(1.0 if binary else float(c))
    return sp.csr_matrix((vals, (rows, cols)), shape=(len(qtexts), len(vocab)), dtype=np.float32)


# ----------------------------------------------------------------------------- weighting
def bm25_matrix(C, k1=0.9, b=0.4):
    """Lucene-style BM25 doc weights: idf * tf*(k1+1)/(tf + k1*(1-b+b*dl/avgdl))."""
    C = C.tocsr().astype(np.float32)
    N = C.shape[0]
    df = np.bincount(C.indices, minlength=C.shape[1])
    idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(C.sum(1)).ravel()
    avgdl = dl.mean()
    W = C.copy()
    rep_dl = np.repeat(dl, np.diff(C.indptr))
    tf = W.data
    W.data = tf * (k1 + 1) / (tf + k1 * (1 - b + b * rep_dl / avgdl)) * idf[W.indices]
    return W, idf


def tfidf_matrix(C):
    """SMART ltc: (1+log tf)*log(N/df), cosine-normalised. Returns (W, idf)."""
    C = C.tocsr().astype(np.float32)
    N = C.shape[0]
    df = np.bincount(C.indices, minlength=C.shape[1])
    idf = np.log(N / np.maximum(df, 1)).astype(np.float32)
    W = C.copy()
    W.data = (1 + np.log(W.data)) * idf[W.indices]
    norms = np.sqrt(np.asarray(W.multiply(W).sum(1)).ravel())
    norms[norms == 0] = 1
    W = sp.diags(1 / norms) @ W
    return W.tocsr(), idf


def search(W, Q, k=1000, batch=128, tie_noise=None, min_score=None):
    """Score all docs: S = Q @ W^T. Returns list of (doc_idx array, score array).

    tie_noise: if given, add uniform noise of that magnitude (seeded) to break ties
    randomly (used for Boolean models whose scores are integers).
    min_score: per-query array; docs scoring below it are not retrieved (Boolean AND).
    """
    rng = np.random.default_rng(SEED)
    WT = W.T.tocsr()
    out = []
    for s in range(0, Q.shape[0], batch):
        Qb = Q[s:s + batch]
        S = np.asarray((Qb @ WT).todense(), dtype=np.float32)
        for i in range(S.shape[0]):
            raw = S[i]
            if min_score is not None:
                valid = np.where(raw >= min_score[s + i] - 1e-6)[0]
            else:
                valid = np.where(raw > 0)[0]
            row = raw
            if tie_noise and len(valid):
                row = raw.copy()
                row[valid] += rng.uniform(0, tie_noise, len(valid)).astype(np.float32)
            kk = min(k, len(valid))
            if kk == 0:
                out.append((np.array([], int), np.array([], np.float32)))
                continue
            sub = row[valid]
            top = np.argpartition(-sub, kk - 1)[:kk]
            top = top[np.argsort(-sub[top], kind="stable")]
            out.append((valid[top], sub[top]))
    return out


def to_run(results, qids, doc_ids, drop_identical=True):
    """Convert search output to {qid: [(docid, score), ...]} (BEIR ignore_identical_ids)."""
    run = {}
    for q, (idx, sc) in zip(qids, results):
        lst = [(doc_ids[i], float(s)) for i, s in zip(idx, sc)]
        if drop_identical:
            lst = [x for x in lst if x[0] != q]
        run[q] = lst
    return run


# ----------------------------------------------------------------------------- evaluation
def evaluate_query(ranked, rels):
    """Per-query metrics. ranked: list of docids; rels: {docid: grade}. Linear gains (trec_eval)."""
    pos = {d: g for d, g in rels.items() if g > 0}
    nrel = len(pos)
    gains = [pos.get(d, 0) for d in ranked]
    # nDCG@10
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains[:10]))
    ideal = sorted(pos.values(), reverse=True)[:10]
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    # MRR@10
    mrr = 0.0
    for i, g in enumerate(gains[:10]):
        if g > 0:
            mrr = 1 / (i + 1)
            break
    # Recall@100, P@10
    hits100 = sum(1 for g in gains[:100] if g > 0)
    rec = hits100 / nrel
    p10 = sum(1 for g in gains[:10] if g > 0) / 10
    # MAP@100 (trec_eval map_cut_100)
    h, ap = 0, 0.0
    for i, g in enumerate(gains[:100]):
        if g > 0:
            h += 1
            ap += h / (i + 1)
    ap /= nrel
    rec1000 = sum(1 for g in gains[:1000] if g > 0) / nrel
    return {"ndcg@10": ndcg, "map@100": ap, "recall@100": rec, "mrr@10": mrr,
            "p@10": p10, "recall@1000": rec1000}


def evaluate_run(run, qrels):
    per = {}
    for q, rels in qrels.items():
        ranked = [d for d, _ in run.get(q, [])]
        per[q] = evaluate_query(ranked, rels)
    return per


def save_run(run, dataset, system, k=1000):
    d = os.path.join(CACHE, "runs")
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d, f"{dataset}__{system}.pkl"), "wb") as f:
        pickle.dump({q: v[:k] for q, v in run.items()}, f, protocol=4)


def load_run(dataset, system):
    with open(os.path.join(CACHE, "runs", f"{dataset}__{system}.pkl"), "rb") as f:
        return pickle.load(f)


def per_query_rows(per, dataset, system):
    return [dict(dataset=dataset, system=system, qid=q, **m) for q, m in per.items()]


In [ ]:
%%writefile /content/proj/src/ce_rerank_colab.py
"""Phase 2 / Agent 4 (E2, E3 at full scale): cross-encoder logits for every evaluated query of the 8 BEIR datasets.

Model: cross-encoder/ms-marco-MiniLM-L-6-v2, max_length 512, fp16 on GPU.
Pool per query = BM25 top-100 (Phase-1 BM25: stem+stop, k1=0.9, b=0.4, identical-id docs dropped)
                 U  H_0.5(D) top-100  U  fused top-10 of H_alpha(D) for alpha in {0, 0.1, ..., 1},
for every dense model D whose Colab run (colab_runs/run_<D>_<ds>.npz, from phase2_dense_runs.ipynb) is on Drive.
Output: <OUT>/ce/ce_<ds>.pkl = {"sample": [all qids], "scores": {(qid, docid): logit}} -- the same format as
phase2/04_improvements/ce_score.py, read automatically by 04_improvements/ce_eval.py. Resumable per dataset.
"""
import os
import pickle
import sys
import time

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import (DATASETS, Index, bm25_matrix, query_matrix, search, to_run, load_dataset)  # noqa: E402

OUT = os.environ.get("OUT", "/content/drive/MyDrive/IR_Lexical_Gap/colab_runs")
CE_OUT = os.path.join(OUT, "ce")
os.makedirs(CE_OUT, exist_ok=True)
TOPK = 100
ALPHAS = np.round(np.linspace(0, 1, 11), 1)


def bm25_run(name):
    idx = Index(name)
    qids = list(idx.qrels)
    C, vocab, an = idx.variant(True, True)
    Q = query_matrix([idx.queries[q] for q in qids], vocab, an)
    W, _ = bm25_matrix(C)
    return to_run(search(W, Q, k=1000), qids, idx.doc_ids)


def dense_run(m, name, doc_ids):
    p = os.path.join(OUT, f"run_{m}_{name}.npz")
    if not os.path.exists(p):
        return None
    z = np.load(p, allow_pickle=False)
    return {str(q): [(doc_ids[i], float(s)) for i, s in zip(ix, ss)] for q, ix, ss in zip(z["qids"], z["idx"], z["scores"])}


def fuse(lb, ld, alphas):
    """Same as common2.convex / run_dense.convex (min-max, absent = 0, stable order B then D)."""
    order, idx, parts = [], {}, []
    for lst in (lb, ld):
        v = np.array([s for _, s in lst], float)
        if len(v):
            lo, hi = v.min(), v.max()
            v = (v - lo) / (hi - lo) if hi > lo else np.ones_like(v)
        parts.append(v)
        for d, _ in lst:
            if d not in idx:
                idx[d] = len(order)
                order.append(d)
    sb, sd = np.zeros(len(order)), np.zeros(len(order))
    for (d, _), v in zip(lb, parts[0]):
        sb[idx[d]] += v
    for (d, _), v in zip(ld, parts[1]):
        sd[idx[d]] += v
    order = np.array(order, dtype=object)
    return {float(a): list(order[np.argsort(-(a * sb + (1 - a) * sd), kind="stable")[:TOPK]]) for a in alphas}


def main():
    from sentence_transformers import CrossEncoder
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512, device=dev)
    if dev == "cuda":
        ce.model.half()
    for name in (sys.argv[1:] or DATASETS):
        out = os.path.join(CE_OUT, f"ce_{name}.pkl")
        doc_ids, texts, queries, qrels = load_dataset(name)
        qids = sorted(qrels)
        if name == "quora":  # 10k queries x ~200 pairs would be ~2M pairs and a ~150 MB file: subsample (stated)
            qn = int(os.environ.get("QUORA_N", "2000"))
            r1, r2 = np.random.default_rng(42), np.random.default_rng(42)
            qids = sorted(set(r1.choice(qids, 150, replace=False).tolist()) |  # = E2 tuning sample S_d
                          set(r2.choice(qids, qn, replace=False).tolist()))
        state = pickle.load(open(out, "rb")) if os.path.exists(out) else {"sample": qids, "scores": {}}
        B = bm25_run(name)
        pool = {q: set(d for d, _ in B.get(q, [])[:TOPK]) for q in qids}
        for m in ("minilm", "bge", "bgebase"):
            D = dense_run(m, name, doc_ids)
            if D is None:
                continue
            for q in qids:
                f = fuse(B.get(q, [])[:1000], D.get(q, [])[:1000], ALPHAS)
                pool[q].update(f[0.5])
                for a in ALPHAS:
                    pool[q].update(f[float(a)][:10])
        pos = {d: i for i, d in enumerate(doc_ids)}
        todo = [(q, d) for q in qids for d in sorted(pool[q]) if (q, d) not in state["scores"] and d in pos]
        todo.sort(key=lambda x: len(texts[pos[x[1]]]) + len(queries[x[0]]))
        print(f"{name}: {len(qids)} queries, {len(todo)} pairs to score", flush=True)
        t0, CH = time.time(), 50000
        for s in range(0, len(todo), CH):
            chunk = todo[s:s + CH]
            sc = ce.predict([(queries[q], texts[pos[d]]) for q, d in chunk], batch_size=128, show_progress_bar=False)
            state["scores"].update({k: float(v) for k, v in zip(chunk, sc)})
            pickle.dump(state, open(out + ".tmp", "wb"))
            os.replace(out + ".tmp", out)
            print(f"  {name}: {s+len(chunk)}/{len(todo)} ({(s+len(chunk))/(time.time()-t0):.0f} pairs/s)", flush=True)
        del B


if __name__ == "__main__":
    main()


In [ ]:
!cd /content/proj && python src/ce_rerank_colab.py 2>&1 | grep -v -i 'warn\|loading weights'

In [ ]:
!cd /content/drive/MyDrive/IR_Lexical_Gap && zip -qr /content/phase2_ce_runs.zip colab_runs/ce && ls -la /content/phase2_ce_runs.zip
from google.colab import files
files.download('/content/phase2_ce_runs.zip')